# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:  
[https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant -U

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We'll list all record sets, their fields, and columns, referencing everything by `@id`.

In [ ]:
# List all record sets, their @id, and the fields and columns they contain
recordsets_info = []
for record_set in dataset.record_sets:
    print(f"Record set: {record_set.name} (@id: {record_set.id})")
    fields = getattr(record_set, 'fields', [])
    print("  Fields and their @id/column associations:")
    for field in fields:
        col_ids = []
        if hasattr(field, 'columns') and field.columns is not None:
            col_ids = [col.id for col in field.columns]
        print(f"    - {field.name} (@id: {field.id}) -> columns: {col_ids if col_ids else 'none'}")
    recordsets_info.append(record_set.id)
    print()
if not recordsets_info:
    print("No record sets found. Check 'dataset.record_sets' for available record sets.")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for further exploration. All references are by `@id`.


In [ ]:
# Build a list of record set IDs from the previous section, or manually provide them if none were listed.
if recordsets_info:
    record_sets = recordsets_info
else:
    # As per FAIR^2, you may need to check the dataset in detail if record_sets is empty
    record_sets = []  # Fill in if missing, e.g., ['your-recordset-id']

dataframes = {}

for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records from record set: {record_set_id}")
        else:
            print(f"No records found for record set: {record_set_id}")
    except Exception as e:
        print(f"Error loading record set {record_set_id}: {e}")

if dataframes:
    example_record_set = list(dataframes.keys())[0]
    print(f"Columns for {example_record_set}:\n", dataframes[example_record_set].columns.tolist())
    dataframes[example_record_set].head()
else:
    print("No dataframes created. Inspect the dataset's record sets and check for records.")

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing: filter records, normalize a numeric field, and do groupings. We will use the first record set found (referenced by its `@id`) and try to pick suitable numeric and categorical fields, if present.


In [ ]:
if dataframes:
    record_set_id = example_record_set
    df = dataframes[record_set_id]

    # Display columns
    print("Available columns:", df.columns.tolist())

    # Attempt to select a numeric field (first float/int-like field)
    possible_numeric = df.select_dtypes(include=['float64', 'int64']).columns
    if len(possible_numeric) == 0:
        # Try to find object columns that can be converted to numeric
        possible_numeric = [col for col in df.columns if pd.to_numeric(df[col], errors='coerce').notnull().any()]

    if len(possible_numeric) > 0:
        numeric_field = possible_numeric[0]
        # Coerce to numeric in case it's not already
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].quantile(0.25)  # Use 25th percentile
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    else:
        print("No numeric field found to analyze.")

    # Attempt to group by a categorical column if available
    possible_categorical = df.select_dtypes(include=['object', 'category']).columns
    group_field = None
    for col in possible_categorical:
        if df[col].nunique() < 20 and col != numeric_field:
            group_field = col
            break

    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Mean {numeric_field} grouped by {group_field}:")
        print(grouped_df.head())
    else:
        print("No suitable categorical field found for grouping.")
else:
    print("No dataframes loaded. Skipping EDA.")

## 5. Visualization
Visualize distributions and relationships between fields using pandas/matplotlib/seaborn, based on the available data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and len(possible_numeric) > 0:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

    # If grouping field exists, plot mean of numeric by group
    if group_field is not None:
        plt.figure(figsize=(10,5))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No suitable numeric field or dataframe for visualization.')

## 6. Conclusion
In this notebook, we've demonstrated how to:
- Load a Croissant-based dataset and its metadata using `mlcroissant`
- Identify record sets, fields, and columns by their unique `@id` references
- Extract tabular data as pandas DataFrames for further processing
- Apply basic exploratory data analysis, filtering, normalization, and grouping
- Visualize distributions and comparisons within the dataset

This workflow can be adapted for other Croissant datasets by simply changing the input schema URL and referencing the correct `@id`s for entities of interest.
